# Solutions – Day 24 Exercises

In [ ]:
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
from PIL import Image
import requests
from io import BytesIO
import numpy as np
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)
pipe.enable_model_cpu_offload()

## Exercise 1: Different input images

In [ ]:
portrait_url = "https://images.pexels.com/photos/614810/pexels-photo-614810.jpeg"
abstract_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/7/77/Abstract_art.jpg/800px-Abstract_art.jpg"

for url, name in [(portrait_url, "portrait"), (abstract_url, "abstract")]:
    img = Image.open(requests.get(url, stream=True).raw).convert("RGB").resize((512,512))
    frames = pipe(img, decode_chunk_size=8, num_frames=25).frames[0]
    export_to_video(frames, f"svd_{name}.mp4", fps=7)
    print(f"Generated {name} video")
print("Portrait tends to have subtle facial motion; abstract may have more chaotic movement.")

## Exercise 2: Smooth loop with cross‑fade

In [ ]:
def smooth_loop(frames, blend_frames=10):
    frames = list(frames)
    last = frames[-1]
    first = frames[0]
    blended = []
    for i in range(blend_frames):
        alpha = i / blend_frames
        blended_frame = Image.blend(last, first, alpha)
        blended.append(blended_frame)
    return frames + blended + frames[::-1]

# Generate base frames (reuse from previous or generate new)
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beach.png"
img = Image.open(requests.get(url, stream=True).raw).convert("RGB").resize((512,512))
frames = pipe(img, decode_chunk_size=8, num_frames=25).frames[0]
looping = smooth_loop(frames, blend_frames=8)
export_to_video(looping, "svd_smooth_loop.mp4", fps=7)
print("Smooth loop created")

## Exercise 3: Frame rate experiment

In [ ]:
for fps in [5, 10, 15]:
    export_to_video(frames, f"svd_{fps}fps.mp4", fps=fps)
    print(f"Exported at {fps} fps")

## Exercise 4: Motion bucket sweep

In [ ]:
for motion in [20, 60, 100, 140, 180, 220]:
    try:
        frames_motion = pipe(img, motion_bucket_id=motion, num_frames=25).frames[0]
        export_to_video(frames_motion, f"svd_motion_{motion}.mp4", fps=7)
        print(f"Motion {motion} done")
    except Exception as e:
        print(f"Motion {motion} failed: {e}")
print("Low motion (<40) almost static; high motion (>180) may cause distortion.")

## Exercise 5: Text‑to‑image + video (two‑step)

In [ ]:
from diffusers import StableDiffusionPipeline
sd = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to(device)
sd.safety_checker = None

prompt = "a dragon flying over a castle, fantasy art"
with torch.no_grad():
    init_img = sd(prompt).images[0].resize((512,512))
display(init_img)

frames = pipe(init_img, num_frames=25).frames[0]
export_to_video(frames, "dragon_castle.mp4", fps=7)
print("Video generated from text prompt via SD + SVD")